# Desk-quarter operations exceptions: grouped count specification search

A specification search on a count response observed repeatedly on the same
units has to settle three things: the response family, a hold-out scheme that
measures error on units the search has not seen, and whether a nominated
interaction improves prediction for those units. This case settles them on a
synthetic panel of trading desks with `SearchSession`, the mboostrs object that
holds the training data and a fixed set of folds, runs one named action at a
time (set a formula, compare candidate formulas on the shared folds, adopt one,
tune the number of boosting steps, freeze, evaluate once on a test set) and
records each result. The search keeps an additive negative binomial model and
adds one term, a separate curve in the number of vendors for desks that run a
legacy technology stack, because that term lowers the out-of-fold error when
both formulas are fitted for 100 boosting steps. When each formula is instead
tuned to its own number of steps, the two have the same out-of-fold error, and
on 28 desks held out from the whole search the additive model has the lower test
mean squared error (20.82 against 21.84; 44.53 for predicting the training
mean). The adopted term compensates for stopping the additive model early.

Each row is one desk in one quarter: 140 desks observed for 28 quarters. The
response `exception_count` is the number of settlement and operations
exceptions the desk recorded in the quarter, a non-negative integer with a long
right tail (training mean 2.14, variance 30.6, maximum 99). Rows that share a
`desk_id` are dependent, so the folds and the test split hold out whole desks.
The predictors are the quarter, the region (AMER, EMEA, APAC), book size
`aum_bn` (assets under management, in billions), `complexity_score`, the share
of overnight settlement `overnight_frac`, the number of vendors `vendor_n`, a
legacy-stack indicator `has_legacy_stack`, staff per billion of assets
`staff_per_bn`, a volatility measure `vol_proxy`, a rate-environment measure
`rate_env` and a utilisation ratio `util_ratio`. Two further columns,
`aum_bn_proxy` and `complexity_z`, are near copies of `aum_bn` and
`complexity_score` (correlations 0.998 and 0.989); they test whether the search
adds a redundant column. `overnight_frac` is missing in 623 of the 3,920 rows.

:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | predict quarterly exception counts for desks outside the training data |
| **Observed response** | `exception_count` (non-negative integer) |
| **Estimand** | test mean squared error of the predicted mean count on held-out desks |
| **Units** | exception events per desk-quarter; squared error in events² |
| **Sampling unit** | one desk in one quarter |
| **Dependence** | 28 quarters per `desk_id`; folds and the test split hold out whole desks |
| **Split unit** | 28 of 140 desks held out for the single test evaluation; four grouped folds on the 112 training desks |
| **Tuning** | number of boosting steps chosen by out-of-fold error on the four grouped folds, after the formula is fixed |
| **Uncertainty** | paired standard error over the four folds for each formula comparison; desk-clustered standard error for the test difference |
| **Claim type** | predictive and methodological, on synthetic data from one generator and one seed |

:::

## Search procedure

The family is `NBinomial`, the negative binomial model for counts with a
log link. Its variance exceeds its mean, which matches counts whose variance
(30.6) is fourteen times their mean (2.14); the Poisson family forces the
variance to equal the mean. The model is fitted by the negative binomial
likelihood, and every comparison is scored by out-of-fold mean squared error of
the predicted mean count (lower is better), the same quantity evaluated on the
test desks.

A comparison fits the current formula and each candidate on the same four
folds, each fold holding out about 28 training desks, for 100 boosting steps at
step length 0.3. The *change* is the candidate's out-of-fold error minus the
current formula's; its standard error is computed from the four paired fold
differences. A candidate is adopted when its change is below minus two standard
errors (the two-standard-error rule); among several that pass, the one with the
largest reduction is adopted.

The search proceeds in five steps:

1. Screening. `pre_scan` fits a tree ensemble (`engine="native"`), a small
   neural network (`engine="neural"`) and both (`engine="dual"`) to rank the
   columns and nominate pairs. These fits only propose; nothing they nominate
   enters the model without a comparison.
2. Baseline. An additive formula with one term per predictor and one column
   from each near-duplicate pair. `bols(x)` is a linear term; `bbs(x, df=4)` is
   a penalised B-spline curve. The family and two shape changes are compared.
3. Interactions. `interaction_screen` ranks pairs by how much of the
   baseline's residual a two-variable fit removes; `pair_candidates` turns the
   leading pairs into formula terms; `tree_candidate` proposes one deep tree
   over all accepted columns. These, with three terms written from the domain
   reading below, are compared against the baseline.
4. Redundancy check. Each near-duplicate column is added to the adopted formula
   and compared.
5. `tune` chooses the number of boosting steps; `freeze` fixes the model;
   `final_evaluate` scores the 28 test desks once.

The domain reading that guided the baseline and the hand-written candidates:
complexity, overnight settlement, a legacy stack and more vendors should raise
exception counts, and more staff per billion of assets should lower them; the
interaction, if any, should be a threshold or a change of slope (for example a
stronger vendor effect on legacy stacks) rather than a product of two linear
terms.

In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import display

import mboostrs as mb
from bench_tables import table
from mboostrs.policy import CostBudget
from mboostrs.search_session import open_search

bench = json.loads(Path("bench_desk_ops_exceptions.json").read_text())
search = bench["search"]
panel = pl.read_parquet("../data/desk_ops_exceptions.parquet")
train = panel.filter(pl.col("split") == "train").drop_nulls()
test = panel.filter(pl.col("split") == "test").drop_nulls()
print(f"Training: {train.height} complete rows from "
      f"{train['desk_id'].n_unique()} desks; test: {test.height} complete rows "
      f"from {test['desk_id'].n_unique()} desks")

Training: 2647 complete rows from 112 desks; test: 650 complete rows from 28 desks


The search uses complete rows only: rows with a missing `overnight_frac` are
dropped before the search, which leaves 2,647 of 3,136 training rows and 650 of
784 test rows. The test error therefore refers to desk-quarters with a recorded
overnight share.

## Family and baseline shapes

In [2]:
fam = search["family_side_compare"]
shape_rows = [{"specification": "family: Poisson", "risk": fam["poisson"],
               "change": fam["poisson"] - fam["nbinomial"], "se": None}]
labels = {"current": "baseline (NBinomial)",
          "linear_complexity": "linear complexity_score",
          "smooth_aum": "smooth aum_bn"}
for row in search["shape_compare"]["rows"]:
    shape_rows.append({"specification": labels[row["specification"]],
                       "risk": row["out-of-fold risk"], "change": row["change"],
                       "se": row["change se"] or None})
table(pl.DataFrame(shape_rows), title="Family and shape comparisons",
      subtitle="Out-of-fold MSE on four grouped folds, 100 steps; lower is better",
      stub="specification", stubhead="Specification",
      labels={"risk": "Out-of-fold MSE", "change": "Change", "se": "Change SE"},
      better={"risk": "min"}, formats={"change": 3, "se": 3},
      notes=["Change is the specification's error minus the NBinomial baseline's.",
             "The family comparison was run without paired fold differences, "
             "so it has no standard error."])

GT(_tbl_data=shape: (4, 4)
┌─────────────────────────┬───────────┬──────────┬──────────┐
│ specification           ┆ risk      ┆ change   ┆ se       │
│ ---                     ┆ ---       ┆ ---      ┆ ---      │
│ str                     ┆ f64       ┆ f64      ┆ f64      │
╞═════════════════════════╪═══════════╪══════════╪══════════╡
│ family: Poisson         ┆ 26.546117 ┆ 7.898164 ┆ null     │
│ baseline (NBinomial)    ┆ 18.647953 ┆ 0.0      ┆ null     │
│ linear complexity_score ┆ 21.191758 ┆ 2.543805 ┆ 0.776658 │
│ smooth aum_bn           ┆ 19.821158 ┆ 1.173204 ┆ 1.341626 │
└─────────────────────────┴───────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10ddb00e0>, _boxhead=Boxhead([ColInfo(var='specification', type=<ColInfoTypeEnum.stub: 2>, column_label='specification', column_align='left', column_width=None, stub_level=0), ColInfo(var='risk', type=<ColInfoTypeEnum.default: 1>, column_label='Out-of-fold MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='Change SE', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10da9cb00>, _spanners=Spanners([]), _heading=Heading(title='Family and shape comparisons', subtitle='Out-of-fold MSE on four grouped folds, 100 steps; lower is better', preheader=None), _stubhead='Specification', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10ddf1280>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10dc9c830>, _source_notes=["Change is the specification's error minus the NBinomial baseline's.", 'The family comparison was run without paired fold differences, so it has no standard error.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b6a7c80>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10daf4ec0>, <great_tables._gt_data.FormatInfo object at 0x10998b680>, <great_tables._gt_data.FormatInfo object at 0x10b6c0110>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10dec0f80>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table',

The negative binomial baseline has out-of-fold error 18.65 against 26.55 for
the same formula under the Poisson family. Replacing the complexity curve by a
line raises the error by 2.54 (standard error 0.78), so the curve is kept.
Replacing the linear book-size term by a curve raises the error by 1.17 with a
standard error of 1.34, so the linear term is kept.

## Interaction candidates

In [3]:
names = {"pair_1": "aum_bn × complexity_score (tensor)",
         "pair_2": "aum_bn × vendor_n (tensor)",
         "pair_3": "complexity_score × vendor_n (tensor)",
         "pair_4": "bbs(vendor_n, by=has_legacy_stack)",
         "tree": "multiway tree, depth 5",
         "complexity_x_overnight": "complexity_score curve × overnight_frac",
         "btree_complexity": "btree(complexity_score)",
         "legacy_x_vendors": "has_legacy_stack × vendor_n (linear)"}
ix_rows = [{"candidate": names[r["specification"]], "risk": r["out-of-fold risk"],
            "change": r["change"], "se": r["change se"]}
           for r in search["interaction_compare"]["rows"]
           if r["specification"] != "current"]
ix = pl.DataFrame(ix_rows).with_columns(
    passes=pl.col("change") < -2 * pl.col("se"))
table(ix, title="Interaction candidates against the additive baseline",
      subtitle="Out-of-fold MSE on four grouped folds; baseline 18.648; lower is better",
      stub="candidate", stubhead="Term added to the baseline",
      labels={"risk": "Out-of-fold MSE", "change": "Change", "se": "Change SE",
              "passes": "Passes two-SE rule"},
      better={"change": "min"}, formats={"risk": 3, "se": 3},
      notes=["The first three tensor candidates and the varying-coefficient term come from "
             "pair_candidates; the depth-5 tree from tree_candidate; the last three "
             "from the domain reading."])

GT(_tbl_data=shape: (8, 5)
┌─────────────────────────────────┬───────────┬───────────┬──────────┬────────┐
│ candidate                       ┆ risk      ┆ change    ┆ se       ┆ passes │
│ ---                             ┆ ---       ┆ ---       ┆ ---      ┆ ---    │
│ str                             ┆ f64       ┆ f64       ┆ f64      ┆ bool   │
╞═════════════════════════════════╪═══════════╪═══════════╪══════════╪════════╡
│ aum_bn × complexity_score (ten… ┆ 21.284694 ┆ 2.63674   ┆ 5.801319 ┆ false  │
│ aum_bn × vendor_n (tensor)      ┆ 19.34054  ┆ 0.692586  ┆ 1.027186 ┆ false  │
│ complexity_score × vendor_n (t… ┆ 18.543058 ┆ -0.104895 ┆ 0.015569 ┆ true   │
│ bbs(vendor_n, by=has_legacy_st… ┆ 18.098922 ┆ -0.549031 ┆ 0.196347 ┆ true   │
│ multiway tree, depth 5          ┆ 3.4511e7  ┆ 3.4511e7  ┆ 3.4472e7 ┆ false  │
│ complexity_score curve × overn… ┆ 18.634818 ┆ -0.013135 ┆ 0.037521 ┆ false  │
│ btree(complexity_score)         ┆ 18.13557  ┆ -0.512384 ┆ 0.158283 ┆ true   │
│ has_legacy_stack × vendor_n (l… ┆ 18.318532 ┆ -0.329422 ┆ 0.205647 ┆ false  │
└─────────────────────────────────┴───────────┴───────────┴──────────┴────────┘, _body=<great_tables._gt_data.Body object at 0x10b6a7c20>, _boxhead=Boxhead([ColInfo(var='candidate', type=<ColInfoTypeEnum.stub: 2>, column_label='candidate', column_align='left', column_width=None, stub_level=0), ColInfo(var='risk', type=<ColInfoTypeEnum.default: 1>, column_label='Out-of-fold MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='Change SE', column_align='right', column_width=None, stub_level=None), ColInfo(var='passes', type=<ColInfoTypeEnum.default: 1>, column_label='Passes two-SE rule', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b6c0140>, _spanners=Spanners([]), _heading=Heading(title='Interaction candidates against the additive baseline', subtitle='Out-of-fold MSE on four grouped folds; baseline 18.648; lower is better', preheader=None), _stubhead='Term added to the baseline', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10dec3ec0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10dec3ef0>, _source_notes=['The first three tensor candidates and the varying-coefficient term come from pair_candidates; the depth-5 tree from tree_candidate; the last three from the domain reading.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10dec0830>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10dec3e00>, <great_tables._gt_data.FormatInfo object at 0x10dec3d40>, <great_tables._gt_data.FormatInfo object at 0x10dec3f50>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10dec3fb0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_

`bbs(vendor_n, by=has_legacy_stack)` is a *varying-coefficient* term: a
penalised curve in the number of vendors multiplied by the legacy-stack
indicator. It is zero for desks without a legacy stack and adds a separate
vendor curve for desks with one, so the vendor effect differs between the two
kinds of desk. It reduces the out-of-fold error by 0.549 (standard error 0.196)
and is adopted. A one-split tree on `complexity_score`, which adds a step to
the complexity curve, reduces the error by 0.512 (standard error 0.158); the
gap between the two is 0.037 with a paired standard error of 0.199, so the
folds do not distinguish them. The complexity-by-vendor tensor passes the rule
with a much smaller reduction (0.105). A tensor term `a %X% b` is a smooth
surface over two variables.

The depth-5 tree over all eleven predictors has an out-of-fold error of about
3.5 × 10⁷: under the log link, a few held-out rows receive extremely large
predicted means. A deep tree term added to a log-link count model can produce
such predictions on desks outside its training folds, and the comparison
rejects it.

## Near-duplicate columns

In [4]:
proxy = [{"specification": {"current": "adopted formula",
                            "add_aum_proxy": "+ bols(aum_bn_proxy)",
                            "add_complexity_z": "+ bols(complexity_z)"}[r["specification"]],
          "risk": r["out-of-fold risk"], "change": r["change"], "se": r["change se"]}
         for r in search["proxy_red_herring_compare"]["rows"]]
table(pl.DataFrame(proxy), title="Adding a near-duplicate column",
      subtitle="Out-of-fold MSE on four grouped folds, 80 steps; lower is better",
      stub="specification", stubhead="Specification",
      labels={"risk": "Out-of-fold MSE", "change": "Change", "se": "Change SE"},
      better={"risk": "min"}, formats={"change": 4, "se": 4})

GT(_tbl_data=shape: (3, 4)
┌──────────────────────┬───────────┬──────────┬──────────┐
│ specification        ┆ risk      ┆ change   ┆ se       │
│ ---                  ┆ ---       ┆ ---      ┆ ---      │
│ str                  ┆ f64       ┆ f64      ┆ f64      │
╞══════════════════════╪═══════════╪══════════╪══════════╡
│ adopted formula      ┆ 18.440865 ┆ 0.0      ┆ 0.0      │
│ + bols(aum_bn_proxy) ┆ 18.444193 ┆ 0.003328 ┆ 0.003328 │
│ + bols(complexity_z) ┆ 18.440865 ┆ 0.0      ┆ 0.0      │
└──────────────────────┴───────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10dec0fe0>, _boxhead=Boxhead([ColInfo(var='specification', type=<ColInfoTypeEnum.stub: 2>, column_label='specification', column_align='left', column_width=None, stub_level=0), ColInfo(var='risk', type=<ColInfoTypeEnum.default: 1>, column_label='Out-of-fold MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='Change SE', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10df0c8f0>, _spanners=Spanners([]), _heading=Heading(title='Adding a near-duplicate column', subtitle='Out-of-fold MSE on four grouped folds, 80 steps; lower is better', preheader=None), _stubhead='Specification', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10dec3650>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10b6c17f0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b6c0080>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10dd3d7c0>, <great_tables._gt_data.FormatInfo object at 0x10b6c0a10>, <great_tables._gt_data.FormatInfo object at 0x10df0cc50>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10df0c0b0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsI

Neither copy improves the adopted formula. `complexity_z` is never selected
by the boosting steps, so its change is exactly zero; `aum_bn_proxy` raises
the error by 0.003. Keeping one column from each near-duplicate pair loses
nothing.

## Test evaluation

The adopted formula and the baseline are each tuned on the grouped folds,
frozen and scored once on the 650 complete test rows.

In [5]:
base = search["baseline_spec"]
adopted = base + " + bbs(vendor_n, by=has_legacy_stack)"

def evaluate(rhs):
    session = open_search(train, "exception_count", family=mb.NBinomial(),
                          score="mse", groups="desk_id", folds=4, seed=20260929,
                          mstop=100, nu=0.3, budget=CostBudget(fits=60))
    session.run("set_spec", rhs=rhs)
    tuned = session.run("tune", start=100).value
    if session.model is None or "freeze" not in session.available_actions():
        session.run("fit")
    session.run("freeze")
    score = session.run("final_evaluate", data=test).summary["score"]
    pred = np.asarray(session.model.predict(test, type="response")).ravel()
    oof = float(np.min(tuned.risk))
    return session.mstop, oof, score, (test["exception_count"].to_numpy() - pred) ** 2

y_mean = train["exception_count"].mean()
const_sq = (test["exception_count"].to_numpy() - y_mean) ** 2
results = {"additive baseline": evaluate(base), "adopted formula": evaluate(adopted)}

/private/tmp/claude-501/-Users-fcela-src-public-github-mboostrs/828991b2-a0bb-4893-ae90-579efe2487d5/scratchpad/docsvenv/lib/python3.12/site-packages/mboostrs/_extrapolation.py:188: ExtrapolationWarning: 2 fold predictions in this call asked 1 learner(s) to predict outside the range they were fitted on. bbs(vendor_n), by=has_legacy_stack, covariate vendor_n: 1 above 10 (worst 0.2 training ranges out). A spline basis continues by its edge slope outside its boundary knots, which the fit does not constrain -- at 0.2 training ranges out that is an assumption, not an estimate. Pass tune_mstop(extrapolation='allow') to accept it, or extrapolation='clip' to hold the effect flat outside the range instead, or extrapolation='error' to refuse. Guide: python -m mboostrs guide modeling-tactics.
  with routine_warnings(stacklevel=3, owner=fn.__name__):


In [6]:
desk = test["desk_id"].to_numpy()

def desk_se(diff):
    # Standard error of a mean difference, clustered by desk.
    totals = pl.DataFrame({"desk": desk, "d": diff}).group_by("desk").agg(
        pl.col("d").sum(), pl.len().alias("n"))
    dbar = diff.mean()
    resid = totals["d"].to_numpy() - totals["n"].to_numpy() * dbar
    g = totals.height
    return float(np.sqrt(g / (g - 1) * np.sum(resid ** 2)) / diff.size)

adopted_sq = results["adopted formula"][3]
rows = [{"model": "training mean", "steps": None, "oof": None, "mse": float(const_sq.mean()),
         "diff": float((const_sq - adopted_sq).mean()), "se": desk_se(const_sq - adopted_sq)}]
for name, (mstop, oof, score, sq) in results.items():
    rows.append({"model": name, "steps": mstop, "oof": oof, "mse": score,
                 "diff": float((sq - adopted_sq).mean()), "se": desk_se(sq - adopted_sq)})
table(pl.DataFrame(rows), title="Test error on 28 held-out desks",
      subtitle="650 complete desk-quarters; mean squared error in events²; lower is better",
      stub="model", stubhead="Model",
      labels={"steps": "Tuned steps", "oof": "Out-of-fold MSE", "mse": "Test MSE",
              "diff": "Excess over adopted", "se": "Desk-clustered SE"},
      better={"mse": "min"}, formats={"oof": 3, "diff": 3, "se": 3, "steps": 0},
      notes=["Out-of-fold MSE is the four-fold error at the tuned number of steps.",
             "Excess over adopted is the model's test MSE minus the adopted formula's; "
             "negative means lower test error than the adopted formula."])

GT(_tbl_data=shape: (3, 6)
┌───────────────────┬───────┬───────────┬───────────┬───────────┬───────────┐
│ model             ┆ steps ┆ oof       ┆ mse       ┆ diff      ┆ se        │
│ ---               ┆ ---   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str               ┆ i64   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════════════╪═══════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ training mean     ┆ null  ┆ null      ┆ 44.525652 ┆ 22.687405 ┆ 17.924453 │
│ additive baseline ┆ 1000  ┆ 17.838305 ┆ 20.817915 ┆ -1.020332 ┆ 0.637522  │
│ adopted formula   ┆ 183   ┆ 17.805542 ┆ 21.838247 ┆ 0.0       ┆ 0.0       │
└───────────────────┴───────┴───────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x10df1fda0>, _boxhead=Boxhead([ColInfo(var='model', type=<ColInfoTypeEnum.stub: 2>, column_label='model', column_align='left', column_width=None, stub_level=0), ColInfo(var='steps', type=<ColInfoTypeEnum.default: 1>, column_label='Tuned steps', column_align='right', column_width=None, stub_level=None), ColInfo(var='oof', type=<ColInfoTypeEnum.default: 1>, column_label='Out-of-fold MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='mse', type=<ColInfoTypeEnum.default: 1>, column_label='Test MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='diff', type=<ColInfoTypeEnum.default: 1>, column_label='Excess over adopted', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='Desk-clustered SE', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b6c04a0>, _spanners=Spanners([]), _heading=Heading(title='Test error on 28 held-out desks', subtitle='650 complete desk-quarters; mean squared error in events²; lower is better', preheader=None), _stubhead='Model', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10cb975c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1206d72f0>, _source_notes=['Out-of-fold MSE is the four-fold error at the tuned number of steps.', "Excess over adopted is the model's test MSE minus the adopted formula's; negative means lower test error than the adopted formula."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1206d7f80>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10b4657c0>, <great_tables._gt_data.FormatInfo object at 0x1206d7dd0>, <great_tables._gt_data.FormatInfo object at 0x10dec1130>, <great_tables._gt_data.FormatInfo object at 0x10dec1730>, <great_tables._gt_data.FormatInfo object at 0x10d227770>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10dec3680>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), tab

Tuning chooses 183 steps for the adopted formula and 1,000 for the additive
baseline. At those step counts the out-of-fold errors are 17.81 and 17.84: the
0.549 reduction measured at 100 steps disappears once the baseline is allowed
to run longer. On the test desks the baseline has error 20.82 and the adopted
formula 21.84, a difference of 1.02 with a desk-clustered standard error of
0.64. Both models halve the error of predicting the training mean (44.53).
The test data do not support the adopted term; the fold comparison credited it
with improvement that more boosting steps on the additive terms also provide.

## Contribution of each term

The term report removes one term at a time from the adopted formula and
records the rise in out-of-fold error ("earns"; larger means the term matters
more).

In [7]:
terms = pl.DataFrame(search["term_report"]["terms"]).select(
    "term", pl.col("earns"), pl.col("se"))
table(terms, title="Rise in out-of-fold error when a term is removed",
      subtitle="Adopted formula, four grouped folds, 100 steps; out-of-fold MSE 18.099",
      stub="term", stubhead="Term", labels={"earns": "Rise in MSE", "se": "SE"},
      formats={"earns": 3, "se": 3},
      notes=["A rise of exactly zero means boosting never selected the term in "
             "100 steps; a negative rise means the error is lower without it."])

GT(_tbl_data=shape: (12, 3)
┌─────────────────────────────────┬───────────┬──────────┐
│ term                            ┆ earns     ┆ se       │
│ ---                             ┆ ---       ┆ ---      │
│ str                             ┆ f64       ┆ f64      │
╞═════════════════════════════════╪═══════════╪══════════╡
│ bbs(complexity_score, df=4)     ┆ 9.502993  ┆ 2.726231 │
│ bols(aum_bn)                    ┆ 5.775752  ┆ 2.641536 │
│ bbs(vendor_n, by=has_legacy_st… ┆ 0.549031  ┆ 0.196347 │
│ bols(staff_per_bn)              ┆ 0.028236  ┆ 0.009741 │
│ bols(has_legacy_stack)          ┆ 0.001108  ┆ 0.017789 │
│ …                               ┆ …         ┆ …        │
│ bols(vol_proxy)                 ┆ 0.0       ┆ 0.0      │
│ bols(rate_env)                  ┆ 0.0       ┆ 0.0      │
│ bols(util_ratio)                ┆ 0.0       ┆ 0.0      │
│ bols(overnight_frac)            ┆ -0.042647 ┆ 0.076395 │
│ bols(region)                    ┆ -0.19747  ┆ 0.990477 │
└─────────────────────────────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x1206d6ed0>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.stub: 2>, column_label='term', column_align='left', column_width=None, stub_level=0), ColInfo(var='earns', type=<ColInfoTypeEnum.default: 1>, column_label='Rise in MSE', column_align='right', column_width=None, stub_level=None), ColInfo(var='se', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1206d7d40>, _spanners=Spanners([]), _heading=Heading(title='Rise in out-of-fold error when a term is removed', subtitle='Adopted formula, four grouped folds, 100 steps; out-of-fold MSE 18.099', preheader=None), _stubhead='Term', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10df0e4e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10df0c260>, _source_notes=['A rise of exactly zero means boosting never selected the term in 100 steps; a negative rise means the error is lower without it.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10df0db20>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1206d7500>, <great_tables._gt_data.FormatInfo object at 0x10df0ecf0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10df0c050>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_borde

The complexity curve (rise 9.50, standard error 2.73) and book size (5.78,
standard error 2.64) carry most of the predictive information. The
legacy-stack vendor curve contributes 0.55 (standard error 0.20) and staff per
billion a small but consistent 0.028 (standard error 0.010). The quarter, the
linear vendor term, volatility, rate environment and utilisation are never
selected in 100 steps; removing region or the overnight share lowers the error
by less than one standard error. Of the domain hypotheses, complexity and book
size are supported by the folds; the legacy-by-vendor curve is supported only
in the comparison at 100 steps, and the overnight share is not supported.

## Implications for the user

- For an overdispersed count response, compare the negative binomial and
  Poisson families on the score you will report; here the choice of family
  changes the out-of-fold error more than any interaction does.
- When rows repeat on a unit, pass that unit as `groups=`. Folds and the test
  split then measure error on new units, which is the error a new desk would
  see.
- Screening fits and `pair_candidates` nominate terms; the fold comparison
  decides. Of eight nominated or hand-written interaction terms, three passed
  the two-standard-error rule at 100 steps, and a deep tree term produced
  extreme predictions.
- A comparison at a fixed number of steps measures a term together with the
  extra capacity it gives an early-stopped model. Before adopting a term,
  compare the candidate and the current formula each at its tuned number of
  steps; here that comparison removes the term's advantage.

## Limitations

- The data are synthetic, from one generator and one seed; the search was run
  once. The fold and test errors carry no uncertainty from repeating the
  search.
- Complete-case rows are used. If the overnight share is missing for reasons
  related to the exception count, the test error does not describe desks with
  a missing share. `open_search(..., na_action="omit")` fits on complete rows
  inside the session instead of requiring the caller to drop them.
- The adopted term and the one-split complexity tree are indistinguishable on
  the folds at 100 steps. Only the adopted term was re-examined at tuned step
  counts.
- The baseline's tuned step count, 1,000, is where its out-of-fold error is
  lowest on a grid that runs to 1,600 steps; the curve is flat near the
  minimum (17.88 at 1,600 steps).
- The screening results (family, shape, interaction and redundancy
  comparisons, and the term report) were recorded with mboostrs 0.1.3. The
  test evaluation is recomputed when the page is built and reproduces the
  recorded test error.

## Reproduction

The data are `docs/book/data/desk_ops_exceptions.parquet`; the recorded search
is `docs/book/cases/bench_desk_ops_exceptions.json`. The search uses grouped
folds with seed 20260929, step length 0.3 and 100 steps per comparison.